<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Elena/NLIDepartmentTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd

In [22]:
df = pd.read_csv("/content/sample_data/department-v2.csv").rename(columns = {"label":"department"})

Load the department dataset, which contains:
* text: job-related descriptions
* label: department category
* Rename label to department for clarity and consistency with later datasets.

# Reformulating Department Classification as an NLI Task

Instead of directly predicting a department label, we reformulate the task as:

Given a job title (premise), does it entail that the job belongs to a specific department (hypothesis)?

This allows us to:
* Use pretrained NLI models
* Leverage semantic reasoning rather than keyword matching

# Construct NLI Training Samples

In [23]:
import random
departments = df["department"].unique().tolist()
rows = []
k = 4
# To balance positive class and negative class. This prevents severe class imbalance

for idx, row in df.iterrows():
  position = row["text"]
  true_dpt = row["department"]
  # Positive (Entailment) Samples
  rows.append(
        {
            "premise":position,
            "hypothesis":"This job belongs to the " + str(true_dpt)+" department.",
            "labels":"entailment"
        }
    )
# Negative (Contradiction) Samples
  neg_dpts = [d for d in departments if d!= true_dpt]
# Randomly select k incorrect departments
  sample_dpts = random.sample(neg_dpts, k = min(k, len(neg_dpts)))
  for neg_dpt in sample_dpts:
    rows.append(
        {
            "premise": position,
            "hypothesis": "This job belongs to the " + str(neg_dpt) +" department.",
            "labels":"contradiction"
        }
    )

NLI_df = pd.DataFrame(rows)

Explanation

Final dataset contains:

* premise
* hypothesis
* labels (entailment or contradiction)

This matches the input format expected by NLI models such as XLM-RoBERTa.

# Evaluation Dataset Construction (CV Data)

In [25]:
# Use the data in the CV dataset as evaluation dataset
eval_df_raw = pd.read_csv("/content/sample_data/df_profiles_cleansed.csv")[["position", "department"]]

* Load LinkedIn CV data
* Use this dataset only for evaluation, not training
* Prevents data leakage

In [27]:
# Convert CV Data into NLI Format
rows=[]
for idx, row in eval_df_raw.iterrows():
  position = row["position"]
  true_dpt = row["department"]
  rows.append(
        {
            "premise":position,
            "hypothesis":"This job belongs to the " + str(true_dpt)+" department.",
            "labels":2  #entailment
        }
    )
eval_df = pd.DataFrame(rows)

Explanation

For evaluation:
* Only create true hypotheses
* Label them as entailment

The model is tested on whether it correctly recognizes true department assignments.

In [28]:
eval_df

,premise,hypothesis,labels
0,Prokurist,This job belongs to the Other department.,2
1,CFO,This job belongs to the Other department.,2
2,Betriebswirtin,This job belongs to the Other department.,2
3,Prokuristin,This job belongs to the Other department.,2
4,CFO,This job belongs to the Other department.,2
...,...,...,...
2610,Justitiar,This job belongs to the Other department.,2
2611,Geschäftsführer,This job belongs to the Other department.,2
2612,Präsidium,This job belongs to the Other department.,2
2613,Rechtsanwalt,This job belongs to the Other department.,2


In [29]:
# Label Encoding
mapping = {"contradiction":0, "neutral":1,"entailment":2}
NLI_df["label_code"] = NLI_df["labels"].map(mapping)

* Transformers require numerical labels
* We follow the standard NLI label convention.
* Although neutral is unused here, it is kept for compatibility.

In [31]:
train_df = NLI_df.drop(columns = "labels").rename(columns = {"label_code":"labels"})

In [32]:
label2id = mapping
id2label = {k:v for v,k in label2id.items()}

# Convert to Hugging Face Datasets

In [33]:
# As required by huggingface trainer
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

# Tokenization with XLM-RoBERTa

XLM-RoBERTa is:
* Multilingual
* Pretrained on the XNLI dataset
* well suited for cross-domain semantic inference

In [35]:
from transformers import AutoTokenizer

model_name = "joeddav/xlm-roberta-large-xnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Encode premise–hypothesis pairs together
def tokenize(batch):
  return tokenizer(batch["premise"],batch["hypothesis"], truncation=True, padding = "max_length", max_length = 106)

train_dataset = train_dataset.map(tokenize, batched = True)
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

eval_dataset = eval_dataset.map(tokenize, batched = True)
eval_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/50725 [00:00<?, ? examples/s]

Map:   0%|          | 0/2615 [00:00<?, ? examples/s]

In [36]:
!pip install evaluate

# Metric Definition

In [37]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
  logits,labels = eval_pred
  predictions = np.argmax(logits, axis = -1)
  return metric.compute(predictions=predictions, references=labels)


In [38]:
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments("test_trainer", report_to="none")

# Model Initialization

In [39]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(label2id),
    id2label=id2label,
    label2id = label2id,
    ignore_mismatched_sizes = True
)

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


* Load pretrained NLI model
* Replace classification head to match our label set
* ignore_mismatched_sizes=True allows safe head reinitialization

# Training and Evaluation

In [40]:

trainer = Trainer(
  model = model,
  args = training_args,
  train_dataset = train_dataset,
  eval_dataset = eval_dataset,
  compute_metrics = compute_metrics
)

trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 2.0105998516082764,
 'eval_model_preparation_time': 0.0127,
 'eval_accuracy': 0.32925430210325046,
 'eval_runtime': 2882.2319,
 'eval_samples_per_second': 0.907,
 'eval_steps_per_second': 0.113}

The accuracy of the zero-shot prediction on department is around 50%, but after training with our data the accuracy dropped to 33%.

Fine-tuning it on a small, synthetically generated NLI dataset introduces label imbalance, task mismatch, and catastrophic forgetting. As a result, the model loses its pretrained semantic knowledge and collapses toward near-random predictions, reducing accuracy from 56% to 33%.